In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 07 · Validación End-to-End — Trust & Safety
# MAGIC
# MAGIC ## Objetivo
# MAGIC
# MAGIC Validar automáticamente la consistencia del pipeline completo:
# MAGIC
# MAGIC Bronze
# MAGIC → DQ
# MAGIC → Silver
# MAGIC → Features
# MAGIC → Trust & Safety
# MAGIC → Risk Consolidation
# MAGIC → Gold
# MAGIC
# MAGIC Si una validación crítica falla, el Job completo termina FAILED.


# COMMAND ----------

from pyspark.sql import functions as F


# COMMAND ----------

# ============================================================
# CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

BRONZE = f"{CATALOG}.yelp_bronze"
SILVER = f"{CATALOG}.yelp_silver"
GOLD = f"{CATALOG}.yelp_gold"
GOV = f"{CATALOG}.yelp_gov"


REVIEW_TABLE = (
    f"{SILVER}.slv_review"
)

FEATURE_TABLE = (
    f"{SILVER}.slv_review_features"
)

EVALUATION_TABLE = (
    f"{SILVER}.slv_review_evaluation"
)

RISK_TABLE = (
    f"{SILVER}.slv_review_risk"
)

RUN_TABLE = (
    f"{GOV}.evaluation_run"
)

RULE_TABLE = (
    f"{GOV}.evaluation_rule"
)

RULE_VERSION_TABLE = (
    f"{GOV}.evaluation_rule_version"
)


print("=" * 72)
print("YELPCONNECT - VALIDACIÓN END TO END")
print("=" * 72)


# COMMAND ----------

# ============================================================
# HELPER
# ============================================================

checks = []


def check(
    name,
    actual,
    expected
):

    passed = (
        actual == expected
    )

    checks.append(
        (
            name,
            str(actual),
            str(expected),
            passed
        )
    )

    icon = (
        "✔"
        if passed
        else "❌"
    )

    print(
        f"{icon} {name}: "
        f"{actual} "
        f"(esperado: {expected})"
    )

    return passed


# COMMAND ----------

# ============================================================
# 1. IDENTIFICAR ÚLTIMO RUN PUBLICABLE
#
# SUCCESS + Risk Consolidation existente.
# ============================================================

risk_runs = (

    spark.table(
        RISK_TABLE
    )

    .select(
        "evaluation_run_id"
    )

    .distinct()
)


latest_run = (

    spark.table(
        RUN_TABLE
    )
    .alias("r")

    .filter(
        F.col(
            "execution_status"
        )
        ==
        "SUCCESS"
    )

    .join(
        risk_runs,
        "evaluation_run_id",
        "inner"
    )

    .orderBy(
        F.desc(
            "started_at"
        )
    )

    .select(
        "evaluation_run_id"
    )

    .first()
)


if latest_run is None:

    raise Exception(
        """
        ❌ No existe Evaluation Run SUCCESS
        con Risk Consolidation.
        """
    )


RUN_ID = (
    latest_run.evaluation_run_id
)


print()
print(
    f"Evaluation Run validado: {RUN_ID}"
)


# COMMAND ----------

# ============================================================
# 2. SILVER REVIEW
# ============================================================

silver_review_count = (

    spark.table(
        REVIEW_TABLE
    )

    .count()
)


silver_review_distinct = (

    spark.table(
        REVIEW_TABLE
    )

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


check(
    "Silver Review - grano",
    silver_review_distinct,
    silver_review_count
)


# COMMAND ----------

# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

feature_stats = (

    spark.table(
        FEATURE_TABLE
    )

    .agg(

        F.count("*")
        .alias(
            "rows"
        ),

        F.countDistinct(
            "review_id"
        )
        .alias(
            "reviews"
        )
    )

    .first()
)


check(
    "Feature rows",
    feature_stats["rows"],
    silver_review_count
)


check(
    "Feature review_id únicos",
    feature_stats["reviews"],
    silver_review_count
)


# COMMAND ----------

# ============================================================
# 4. REGLAS ACTIVAS
# ============================================================

active_rules = (

    spark.table(
        RULE_TABLE
    )
    .alias("r")

    .filter(
        F.col(
            "r.active_flag"
        )
        ==
        True
    )

    .join(

        spark.table(
            RULE_VERSION_TABLE
        )
        .alias("v")

        .filter(
            F.col(
                "v.status"
            )
            ==
            "ACTIVE"
        ),

        "rule_id",

        "inner"
    )

    .select(
        "rule_id",
        "rule_version_id"
    )
)


active_rule_count = (
    active_rules.count()
)


print(
    f"✔ Reglas ACTIVE: "
    f"{active_rule_count}"
)


# COMMAND ----------

# ============================================================
# 5. TRUST & SAFETY EVALUATION
# ============================================================

evaluation_stats = (

    spark.table(
        EVALUATION_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        RUN_ID
    )

    .agg(

        F.count("*")
        .alias(
            "rows"
        ),

        F.countDistinct(
            "review_id"
        )
        .alias(
            "reviews"
        ),

        F.countDistinct(
            "rule_version_id"
        )
        .alias(
            "rules"
        )
    )

    .first()
)


expected_evaluations = (

    silver_review_count
    *
    active_rule_count
)


check(
    "Reviews evaluadas T&S",
    evaluation_stats["reviews"],
    silver_review_count
)


check(
    "Reglas evaluadas",
    evaluation_stats["rules"],
    active_rule_count
)


check(
    "Evaluaciones T&S",
    evaluation_stats["rows"],
    expected_evaluations
)


# COMMAND ----------

# ============================================================
# 6. RISK CONSOLIDATION
# ============================================================

risk_stats = (

    spark.table(
        RISK_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        RUN_ID
    )

    .agg(

        F.count("*")
        .alias(
            "rows"
        ),

        F.countDistinct(
            "review_id"
        )
        .alias(
            "reviews"
        ),

        F.sum(

            F.when(
                F.col(
                    "requires_investigation"
                ),
                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "investigation"
        )
    )

    .first()
)


check(
    "Risk rows",
    risk_stats["rows"],
    silver_review_count
)


check(
    "Risk review_id únicos",
    risk_stats["reviews"],
    silver_review_count
)


# COMMAND ----------

# ============================================================
# 7. GOLD REVIEW RISK
# ============================================================

gold_risk_stats = (

    spark.table(
        f"{GOLD}.gld_review_risk"
    )

    .agg(

        F.count("*")
        .alias(
            "rows"
        ),

        F.countDistinct(
            "review_id"
        )
        .alias(
            "reviews"
        )
    )

    .first()
)


check(
    "Gold Review Risk",
    gold_risk_stats["rows"],
    risk_stats["rows"]
)


check(
    "Gold Review Risk únicos",
    gold_risk_stats["reviews"],
    risk_stats["reviews"]
)


# COMMAND ----------

# ============================================================
# 8. GOLD FACT
# ============================================================

gold_fact_count = (

    spark.table(
        f"{GOLD}.gld_fact_review_evaluation"
    )

    .count()
)


check(
    "Gold Fact Evaluation",
    gold_fact_count,
    evaluation_stats["rows"]
)


# COMMAND ----------

# ============================================================
# 9. INVESTIGATION QUEUE
# ============================================================

queue_count = (

    spark.table(
        f"{GOLD}.gld_investigation_queue"
    )

    .count()
)


check(
    "Investigation Queue",
    queue_count,
    risk_stats["investigation"]
)


# COMMAND ----------

# ============================================================
# 10. OVERVIEW GOLD
# ============================================================

overview = (

    spark.table(
        f"{GOLD}.gld_kpi_trust_safety_overview"
    )

    .first()
)


check(
    "Overview Reviews",
    overview["reviews_evaluated"],
    silver_review_count
)


check(
    "Overview Investigación",
    overview["reviews_for_investigation"],
    queue_count
)


# COMMAND ----------

# ============================================================
# 11. PRIORIDADES
# ============================================================

priority_stats = (

    spark.table(
        f"{GOLD}.gld_review_risk"
    )

    .groupBy(
        "investigation_priority"
    )

    .count()
)


display(
    priority_stats
)


priority_total = (

    priority_stats

    .agg(
        F.sum(
            "count"
        ).alias(
            "total"
        )
    )

    .first()
    .total
)


check(
    "Total prioridades",
    priority_total,
    silver_review_count
)


# COMMAND ----------

# ============================================================
# 12. RESULTADOS
# ============================================================

validation_df = (

    spark.createDataFrame(

        checks,

        [
            "validation",
            "actual",
            "expected",
            "passed"
        ]
    )
)


display(
    validation_df
)


# COMMAND ----------

# ============================================================
# 13. ACCEPTANCE GATE
# ============================================================

failed_checks = (

    validation_df

    .filter(
        F.col(
            "passed"
        )
        ==
        False
    )

    .count()
)


print()
print("=" * 72)
print("RESULTADO END-TO-END")
print("=" * 72)

print(
    f"Checks ejecutados : "
    f"{len(checks)}"
)

print(
    f"Checks fallidos   : "
    f"{failed_checks}"
)


if failed_checks > 0:

    print()
    print(
        "❌ PIPELINE END-TO-END RECHAZADO"
    )

    raise Exception(
        f"""
        {failed_checks} validaciones
        end-to-end fallaron.
        """
    )


print()
print(
    "✔ PIPELINE END-TO-END APROBADO"
)

print()
print(
    "La capa Gold está lista "
    "para consumo Power BI."
)

print("=" * 72)